# Sanity Checks (Pre-Join)

Before joining datasets, we validate data integrity, key consistency, and structural assumptions.

This ensures that developer profiles (contacts) correctly map to engagement activity (activities), which is critical for downstream analysis like clustering and journey tracking.

The datasets are expected to join on:

- `contacts.developer_id`
- `activities.dev_contact` (foreign key) :contentReference[oaicite:0]{index=0}

In [8]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

con = duckdb.connect("developer_project.duckdb")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [9]:
con.execute("""SHOW TABLES""").fetchdf()

,name
0,activity_base
1,activity_clean
2,activity_final
3,activity_raw
4,activity_score_mapping_clean
5,activity_score_mapping_raw
6,activity_work
7,contact_clean
8,contact_clean_backup_before_supplement
9,contact_final


## 1. Dataset Overview

We begin by checking the size of each table to understand scale and confirm that the data loaded correctly into DuckDB.

In [10]:
con.execute("""
SELECT 'contact_final' AS table_name, COUNT(*) AS row_count
FROM contact_final

UNION ALL

SELECT 'activity_final' AS table_name, COUNT(*) AS row_count
FROM activity_final
""").fetchdf()

,table_name,row_count
0,contact_final,9381490
1,activity_final,69347501


# Sanity Checks (Pre-Join)

Before joining datasets, we validate data integrity, key consistency, and structural assumptions.

This ensures that developer profiles in `contact_final` correctly map to engagement activity in `activity_final`, which is critical for downstream analysis such as clustering and developer journey tracking.

The datasets are expected to join on:

- `contact_final.developer_id`
- `activity_final.dev_contact`

## 2. Join Key Null Checks

We first verify that the key fields needed for joining are populated.

Missing identifiers would lead to dropped matches and incomplete developer activity histories.

In [11]:
con.execute("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN developer_id IS NULL THEN 1 ELSE 0 END) AS null_developer_id
FROM contact_final
""").fetchdf()

,total_rows,null_developer_id
0,9381490,0.0


In [12]:
con.execute("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN dev_contact IS NULL THEN 1 ELSE 0 END) AS null_dev_contact
FROM activity_final
""").fetchdf()

,total_rows,null_dev_contact
0,69347501,0.0


## 3. Contact Key Uniqueness

Each developer profile should have a unique `developer_id`.

If duplicate developer IDs exist in the contact table, the join may create unintended many-to-many relationships.

In [13]:
con.execute("""
SELECT COUNT(*) AS duplicate_developer_ids
FROM (
    SELECT developer_id
    FROM contact_final
    GROUP BY developer_id
    HAVING COUNT(*) > 1
) t
""").fetchdf()

,duplicate_developer_ids
0,0


In [14]:
con.execute("""
SELECT developer_id, COUNT(*) AS row_count
FROM contact_final
GROUP BY developer_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").fetchdf()

,developer_id,row_count


## 4. Activity Key Coverage

We check whether all activity records reference a developer ID that exists in the contact table.

This helps estimate how many activity rows would fail to join.

In [15]:
con.execute("""
SELECT COUNT(*) AS unmatched_activity_rows
FROM activity_final a
LEFT JOIN contact_final c
    ON a.dev_contact = c.developer_id
WHERE c.developer_id IS NULL
""").fetchdf()

,unmatched_activity_rows
0,2393


In [16]:
con.execute("""
SELECT COUNT(DISTINCT a.dev_contact) AS unmatched_activity_ids
FROM activity_final a
LEFT JOIN contact_final c
    ON a.dev_contact = c.developer_id
WHERE c.developer_id IS NULL
""").fetchdf()

,unmatched_activity_ids
0,18


## 5. Contacts Without Activity

Not every developer profile is expected to have engagement activity.

This check shows how many contacts would remain unmatched from the activity side.

In [17]:
con.execute("""
SELECT COUNT(*) AS contacts_without_activity
FROM contact_final c
LEFT JOIN activity_final a
    ON c.developer_id = a.dev_contact
WHERE a.dev_contact IS NULL
""").fetchdf()

,contacts_without_activity
0,1721230


## 6. Expected Join Cardinality

The expected relationship is one developer to many activities.

We summarize activity counts per developer to confirm that assumption.

In [18]:
con.execute("""
SELECT 
    COUNT(*) AS developers_with_activity,
    MIN(activity_count) AS min_activities,
    MAX(activity_count) AS max_activities,
    AVG(activity_count) AS avg_activities
FROM (
    SELECT dev_contact, COUNT(*) AS activity_count
    FROM activity_final
    WHERE dev_contact IS NOT NULL
    GROUP BY dev_contact
) t
""").fetchdf()

,developers_with_activity,min_activities,max_activities,avg_activities
0,7660278,1,2027375,9.05287


In [19]:
con.close()